# 01 Loading Real Timetable & Location Data
**Project:** A Predictive Analytics System for Detecting Bus Bunching Events
**Data:** Stagecoach Cumbria & North Lancashire network (BODS)


In [1]:
import os, sys
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['HADOOP_HOME'] = r'C:\Hadoop'

In [2]:
import os
import pandas as pd

os.makedirs("data", exist_ok=True)

TIMETABLE_CSV = r"C:\Users\ACER\OneDrive\Documents\Big Data Programming Project Coursework\data\timetable_clean.csv"              
ROUTES_HEADWAY_CSV =r"C:\Users\ACER\OneDrive\Documents\Big Data Programming Project Coursework\data\routes_scheduled_headway.csv"   
LOCATION_LOG_PATH = r"C:\Users\ACER\OneDrive\Documents\Big Data Programming Project Coursework\bus_locations_log.csv"          


## 1.1 Load the real timetable data (already parsed — just copy through with a sanity check)

In [3]:
timetable_df = pd.read_csv(TIMETABLE_CSV, low_memory=False)
print(f"Timetable rows: {len(timetable_df)}")
print(f"Distinct routes: {timetable_df['line_ref'].nunique()}")
print(f"Null stop_point_ref: {timetable_df['stop_point_ref'].isna().sum()}")
print(f"Null scheduled_time_sec: {timetable_df['scheduled_time_sec'].isna().sum()}")

timetable_df.to_csv(r"C:\Users\ACER\OneDrive\Documents\Big Data Programming Project Coursework\data\timetable_clean.csv", index=False)
print("Saved data/timetable_clean.csv")
timetable_df.head()


Timetable rows: 476087
Distinct routes: 106
Null stop_point_ref: 0
Null scheduled_time_sec: 0
Saved data/timetable_clean.csv


,service_code,line_ref,operator_noc,vehicle_journey_code,direction,route_ref,stop_sequence,stop_point_ref,stop_name,scheduled_time_sec,source_file
0,PC0002407:332,6,SCCU,VJ1,inbound,RT4,0,090010020517,Barrow Town Hall Stop A,20400,BA06-None--SCCU-NWBA-2025-08-14-6_6A_6C_X6_Sum...
1,PC0002407:332,6,SCCU,VJ1,inbound,RT4,1,090010021163,Ramsden Square Stop K,20460,BA06-None--SCCU-NWBA-2025-08-14-6_6A_6C_X6_Sum...
2,PC0002407:332,6,SCCU,VJ1,inbound,RT4,2,090010020529,Magistrates Courts,20580,BA06-None--SCCU-NWBA-2025-08-14-6_6A_6C_X6_Sum...
3,PC0002407:332,6,SCCU,VJ1,inbound,RT4,3,090010020532,West View Road,20640,BA06-None--SCCU-NWBA-2025-08-14-6_6A_6C_X6_Sum...
4,PC0002407:332,6,SCCU,VJ1,inbound,RT4,4,090010020535,Whitehouse Hotel,20700,BA06-None--SCCU-NWBA-2025-08-14-6_6A_6C_X6_Sum...


,service_code,line_ref,operator_noc,vehicle_journey_code,direction,route_ref,stop_sequence,stop_point_ref,stop_name,scheduled_time_sec,source_file
0,PC0002407:332,6,SCCU,VJ1,inbound,RT4,0,090010020517,Barrow Town Hall Stop A,20400,BA06-None--SCCU-NWBA-2025-08-14-6_6A_6C_X6_Sum...
1,PC0002407:332,6,SCCU,VJ1,inbound,RT4,1,090010021163,Ramsden Square Stop K,20460,BA06-None--SCCU-NWBA-2025-08-14-6_6A_6C_X6_Sum...
2,PC0002407:332,6,SCCU,VJ1,inbound,RT4,2,090010020529,Magistrates Courts,20580,BA06-None--SCCU-NWBA-2025-08-14-6_6A_6C_X6_Sum...
3,PC0002407:332,6,SCCU,VJ1,inbound,RT4,3,090010020532,West View Road,20640,BA06-None--SCCU-NWBA-2025-08-14-6_6A_6C_X6_Sum...
4,PC0002407:332,6,SCCU,VJ1,inbound,RT4,4,090010020535,Whitehouse Hotel,20700,BA06-None--SCCU-NWBA-2025-08-14-6_6A_6C_X6_Sum...


## 1.2 Load the real scheduled-headway-per-route data (already computed)

In [4]:
routes_headway = pd.read_csv(ROUTES_HEADWAY_CSV)
print(f"Routes: {len(routes_headway)}")
routes_headway.to_csv(r"C:\Users\ACER\OneDrive\Documents\Big Data Programming Project Coursework\data\routes_scheduled_headway.csv", index=False)
print("Saved data/routes_scheduled_headway.csv")
routes_headway.head(10)


Routes: 106
Saved data/routes_scheduled_headway.csv


,line_ref,route_name,scheduled_headway_sec,n_departures
0,1,Route 1,339.355930,755
1,10,Route 10,1508.108108,65
2,100,Route 100,767.033784,315
3,104,Route 104,1663.762447,195
4,109,Route 109,1659.720280,112
5,11,Route 11,1591.666667,62
6,111,Route 111,407.766319,330
7,125,Route 125,478.524590,371
8,127,Route 127,1982.727273,69
9,18,Route 18,3600.000000,9


## 1.3 Load and validate the real AVL location log from your polling script

Polling script re-fetches the live feed every ~20 seconds. Since many vehicles only send
a new GPS fix every 30-60 seconds, the same recorded_at_time often appears across several
consecutive polls for the same vehicle — these are genuine repeated observations of the same
real-world position, not bad data, but they are redundant for our purposes, so we deduplicate
on vehicles_ref , recorded_at_time to keep one row per genuinely distinct real observation.


In [5]:
def validate_location_log(path):
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"{path} not found. Run poll_locations.py first (see project setup) to produce "
            f"this file, then re-run this cell -- no synthetic fallback is used in this project."
        )
    df = pd.read_csv(path)
    before = len(df)
    df = df.dropna(subset=["line_ref", "vehicle_ref", "recorded_at_time", "latitude", "longitude"])
    df = df.drop_duplicates(subset=["vehicle_ref", "recorded_at_time"])
    print(f"Validated {path}: {before} raw rows -> {len(df)} distinct real observations "
          f"after dropping nulls/duplicates")
    return df

location_df = validate_location_log(LOCATION_LOG_PATH)
location_df.to_csv("data/location_raw_validated.csv", index=False)
print("Saved data/location_raw_validated.csv")
location_df.head()


Validated C:\Users\ACER\OneDrive\Documents\Big Data Programming Project Coursework\bus_locations_log.csv: 27439 raw rows -> 762 distinct real observations after dropping nulls/duplicates
Saved data/location_raw_validated.csv


,poll_timestamp,recorded_at_time,line_ref,vehicle_ref,operator_ref,latitude,longitude,bearing
0,2026-07-29T09:05:56.291253,2026-07-28T21:35:13+00:00,X5,SCCU-10014,SCCU,54.667065,-2.755720,96
1,2026-07-29T09:05:56.291253,2026-07-28T20:08:36+00:00,104,SCCU-10016,SCCU,54.894077,-2.932554,0
2,2026-07-29T09:05:56.291253,2026-07-28T17:40:42+00:00,79,SCCU-10018,SCCU,54.894203,-2.932171,132
3,2026-07-29T09:05:56.291253,2026-07-28T18:27:18+00:00,1A,SCCU-10019,SCCU,54.050888,-2.800404,42
4,2026-07-29T09:05:56.291253,2026-07-28T18:35:52+00:00,2X,SCCU-10020,SCCU,54.035912,-2.897562,180


## 1.4 Route-name overlap check between timetable and location data

Confirms two real data sources actually reference the same routes, which is required between scheduled and observed headway to produce meaningful results.


In [5]:
timetable_routes = set(routes_headway["line_ref"].astype(str).unique())
location_routes = set(location_df["line_ref"].astype(str).unique())
overlap = timetable_routes & location_routes

print(f"Routes in timetable: {len(timetable_routes)}")
print(f"Routes with live AVL data: {len(location_routes)}")
print(f"Routes present in BOTH (usable for bunching analysis): {len(overlap)}")
print(f"\nOverlapping routes: {sorted(overlap)}")


Routes in timetable: 106
Routes with live AVL data: 109
Routes present in BOTH (usable for bunching analysis): 72

Overlapping routes: ['1', '100', '104', '109', '11', '111', '125', '127', '1A', '2', '22', '280', '29', '29A', '2A', '2X', '3', '30', '300', '32', '3A', '4', '40', '400', '41', '41A', '42', '43A', '44', '45', '5', '505', '508', '509', '51', '516', '52', '530', '534', '55', '555', '563', '567', '59', '599', '6', '60', '600', '61', '62', '63', '63A', '67', '68', '685', '6A', '6C', '7', '755', '77', '77A', '78', '81', '88', '89', '9', '93', 'P1', 'X2', 'X4', 'X5', 'X6']
